# Módulo VIII — Dos *Transformers* às *Large Language Models* (LLMs)

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

## Mão na Massa: o Mecanismo por Dentro (opcional)

In [ ]:
import sys
sys.path.insert(0, "..")          # permite importar src/ a partir de notebooks/
import numpy as np
import matplotlib.pyplot as plt
from src.dados_sinteticos_m8_m9 import corpus_bpe, palavras_embeddings

np.set_printoptions(precision=3, suppress=True)

### *Scaled Dot-Product Attention* do Zero

In [ ]:
def softmax(x, eixo=-1):
    x = x - x.max(axis=eixo, keepdims=True)      # estabilidade numérica
    e = np.exp(x)
    return e / e.sum(axis=eixo, keepdims=True)

def atencao(Q, K, V, mascara=None):
    """Attention(Q,K,V) = softmax(Q K^T / sqrt(d_k)) V."""
    d_k = Q.shape[-1]
    pontuacoes = Q @ K.T / np.sqrt(d_k)             # (n, n): alinhamento de cada par
    if mascara is not None:
        pontuacoes = np.where(mascara, pontuacoes, -1e9)   # proibido -> peso ~ 0
    pesos = softmax(pontuacoes)                     # cada linha soma 1
    return pesos @ V, pesos

tokens = ["o", "gato", "comeu", "o", "peixe"]
n, d_modelo, d_k = len(tokens), 8, 4
rng = np.random.default_rng(42)
tabela = rng.normal(size=(4, d_modelo))             # 1 vetor por palavra distinta (sorteado)
X = tabela[[0, 1, 2, 0, 3]]                         # "o" aparece 2x: mesmo vetor nas posições 0 e 3

# projeções lineares: no modelo real, W^Q, W^K, W^V são APRENDIDAS
Wq, Wk, Wv = (rng.normal(size=(d_modelo, d_k)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv

saida, pesos = atencao(Q, K, V)
print("formas:", Q.shape, K.shape, V.shape, "->", saida.shape)
print(pesos)
print("soma de cada linha:", pesos.sum(axis=1))

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(pesos, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n), tokens)
ax.set_yticks(range(n), tokens)
ax.set_xlabel("token consultado (key)")
ax.set_ylabel("token que consulta (query)")
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{pesos[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="peso de atenção")
plt.show()

### Por que Dividir por $\sqrt{d_k}$?

In [ ]:
d_grande = 256
Qg, Kg = rng.normal(size=(500, d_grande)), rng.normal(size=(500, d_grande))
bruto = (Qg * Kg).sum(axis=1)                       # produtos escalares sem escala
escalado = bruto / np.sqrt(d_grande)
print("desvio-padrão sem escala:", round(bruto.std(), 2), "| com escala:", round(escalado.std(), 2))

z = rng.normal(size=8)                              # 8 pontuações quaisquer
for rotulo, fator in [("pontuações originais", 1), ("pontuações x 16", 16)]:
    print(f"{rotulo:22s} -> maior peso da softmax = {softmax(z * fator).max():.3f}")

### Máscara Causal: Impedindo de Olhar para o Futuro

In [ ]:
mascara_causal = np.tril(np.ones((n, n), dtype=bool))   # True onde pode olhar (passado e próprio)
print(mascara_causal.astype(int))

_, pesos_causais = atencao(Q, K, V, mascara=mascara_causal)
print(pesos_causais)
print("peso acima da diagonal (futuro):", pesos_causais[~mascara_causal].max())

### Atenção Multi-Cabeça Simplificada

In [ ]:
def atencao_multicabeca(X, h, rng):
    """Versão simplificada: h cabeças com d_k = d_modelo / h, concatenação e W^O."""
    d_modelo = X.shape[1]
    d_k = d_modelo // h
    cabecas, todos_pesos = [], []
    for _ in range(h):
        Wq, Wk, Wv = (rng.normal(size=(d_modelo, d_k)) for _ in range(3))
        saida, pesos = atencao(X @ Wq, X @ Wk, X @ Wv)
        cabecas.append(saida)
        todos_pesos.append(pesos)
    Wo = rng.normal(size=(h * d_k, d_modelo))
    return np.concatenate(cabecas, axis=1) @ Wo, todos_pesos   # Concat(head_1..h) W^O

rng_mh = np.random.default_rng(0)
saida_mh, pesos_cabecas = atencao_multicabeca(X, h=2, rng=rng_mh)
print("entrada:", X.shape, "-> saída:", saida_mh.shape)
for i, p in enumerate(pesos_cabecas, 1):
    print(f"cabeça {i}: atenção do token '{tokens[1]}' ->", p[1])   # distribuições diferentes

### Codificação Posicional Senoidal

In [ ]:
def codificacao_posicional(n_pos, d_modelo):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d_modelo // 2)[None, :]
    angulos = pos / (10000 ** (2 * i / d_modelo))
    PE = np.zeros((n_pos, d_modelo))
    PE[:, 0::2] = np.sin(angulos)       # dimensões pares: seno
    PE[:, 1::2] = np.cos(angulos)       # dimensões ímpares: cosseno
    return PE

PE = codificacao_posicional(50, 16)
print(PE[:3, :6])

X_pos = X + codificacao_posicional(n, d_modelo)         # embedding + posição
print("sem posição, os dois 'o' têm o mesmo vetor?  ->", np.allclose(X[0], X[3]))
print("com posição, ainda têm o mesmo vetor?        ->", np.allclose(X_pos[0], X_pos[3]))

_, pesos_pos = atencao(X_pos @ Wq, X_pos @ Wk, X_pos @ Wv)
print("atenção dos dois 'o' sem posição:", np.allclose(pesos[0], pesos[3]), "(linhas idênticas)")
print("atenção dos dois 'o' com posição:", np.allclose(pesos_pos[0], pesos_pos[3]), "(linhas distintas)")

### Um Tokenizador BPE de Brinquedo

In [ ]:
def treinar_bpe(palavras, n_fusoes):
    """BPE de brinquedo: começa em caracteres e funde o par adjacente mais frequente."""
    corpus = [tuple(p) + ("</w>",) for p in palavras]       # "baixo" -> b a i x o </w>
    fusoes = []
    for _ in range(n_fusoes):
        contagem = {}
        for palavra in corpus:
            for par in zip(palavra, palavra[1:]):
                contagem[par] = contagem.get(par, 0) + 1
        melhor = max(contagem, key=lambda par: (contagem[par], par))   # desempate fixo
        fusoes.append(melhor)
        novo = []
        for palavra in corpus:
            saida, i = [], 0
            while i < len(palavra):
                if i < len(palavra) - 1 and (palavra[i], palavra[i + 1]) == melhor:
                    saida.append(palavra[i] + palavra[i + 1]); i += 2
                else:
                    saida.append(palavra[i]); i += 1
            novo.append(tuple(saida))
        corpus = novo
        print(f"fusão {len(fusoes)}: {melhor}  (aparece {contagem[melhor]}x)")
    return fusoes, corpus

fusoes, corpus_final = treinar_bpe(corpus_bpe(), n_fusoes=6)
print(sorted(set(corpus_final)))

In [ ]:
def tokenizar(palavra, fusoes):
    simbolos = list(palavra) + ["</w>"]
    for a, b in fusoes:                                    # aplica as fusões na ordem aprendida
        i = 0
        while i < len(simbolos) - 1:
            if simbolos[i] == a and simbolos[i + 1] == b:
                simbolos[i:i + 2] = [a + b]
            else:
                i += 1
    return simbolos

for p in ["baixo", "caixas", "peixes", "baixinho"]:
    print(f"{p:9s} -> {tokenizar(p, fusoes)}")

### Similaridade do Cosseno entre *Embeddings*

In [ ]:
emb = palavras_embeddings()

def cosseno(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

for a, b in [("gato", "felino"), ("gato", "cachorro"), ("gato", "bolsa"), ("rei", "rainha")]:
    print(f"cos({a}, {b}) = {cosseno(emb[a], emb[b]):.3f}")

analogia = emb["rei"] - emb["homem"] + emb["mulher"]
ranking = sorted(((cosseno(analogia, v), p) for p, v in emb.items()
                  if p not in ("rei", "homem", "mulher")), reverse=True)
print("rei - homem + mulher ~", [(p, round(float(s), 3)) for s, p in ranking[:3]])